# Synthetic Data — Dev Log

## Objetivo

Gera dados sintéticos determinísticos (CPF com dígito verificador módulo 11
real, nomes/e-mails/telefones fictícios) para testar outros módulos sem
usar PII real. `synthetic=True` fixo no schema — impossível confundir com
dado real por engano.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.pii_detection.detector import detect
from core.synthetic_data.generator import generate_dataset

dataset = generate_dataset(3, seed=2026)
for record in dataset:
    print(f"  {record.name} | {record.cpf} | {record.email} | {record.phone}")
print()
text = f"Nome: {dataset[0].name}, CPF: {dataset[0].cpf}"
result = detect(text)
print(f"pii_detection.detect() no primeiro registro sintético: {[f.entity_type for f in result.findings]}")

  João Silva | 275.938.584-13 | joão.silva875@dados-ficticios.test | (11) 94113-8105
  Bruno Souza | 301.252.088-04 | bruno.souza358@exemplo-sintetico.test | (51) 91617-1989
  João Pereira | 408.803.102-43 | joão.pereira733@dados-ficticios.test | (61) 93419-8737

pii_detection.detect() no primeiro registro sintético: ['NOME', 'CPF']


O CPF sintético é reconhecido pelo motor real de `pii_detection` com alta
confiança — prova de que o dígito verificador é estruturalmente válido
(módulo 11 real), não um número aleatório que "parece" um CPF.

## Testes e Handoff

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/synthetic_data/tests -v
```

11/11 testes passando, incluindo round-trip completo contra `pii_detection.detect()`.